# Tutorial 14: Unified Embedding with embed_adata

The `embed_adata()` method is the **single entry point** for computing
both **cell-level** and **perturbation-level** embeddings on an AnnData
object in one call.

- **Cell embeddings** come from expression data (PCA, scVI, scGPT, Geneformer, ...)
- **Perturbation embeddings** come from perturbation annotations in `.obs`
  (gene symbols -> DNA/protein embeddings, SMILES -> molecule embeddings)

All results are stored in `.obsm` with a consistent `X_` prefix.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
import anndata as ad
import numpy as np
import scipy.sparse as sp
import pandas as pd

from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

## 1. Load a Perturb-seq Dataset

We use a small subset of a Perturb-seq dataset. The key column is
the perturbation annotation in `.obs` (e.g. gene symbols or SMILES).

In [ ]:
# Load example dataset (adjust path as needed)
# adata = ad.read_h5ad("path/to/perturbseq.h5ad")

# For demonstration, create a synthetic dataset
rng = np.random.default_rng(42)
n_cells, n_genes = 500, 2000
X = sp.random(n_cells, n_genes, density=0.1, format="csr",
              random_state=42, dtype=np.float32)
X.data = np.abs(X.data) * 100  # make count-like

gene_names = [f"Gene_{i}" for i in range(n_genes)]
perturbations = rng.choice(
    ["TP53", "BRCA1", "EGFR", "MYC", "control"],
    size=n_cells,
)

adata = ad.AnnData(
    X=X,
    obs=pd.DataFrame({"perturbation": perturbations}),
    var=pd.DataFrame(index=gene_names),
)
adata.obs.index = [f"cell_{i}" for i in range(n_cells)]

print(adata)
print(f"Perturbations: {adata.obs['perturbation'].value_counts().to_dict()}")

## 2. Cell Embeddings Only

When you only need expression-based embeddings, pass `cell_models`
without `perturbation_models`.

In [ ]:
result = embedder.embed_adata(
    adata,
    cell_models=["pca"],
    preprocessing="standard",
    n_pca_components=30,
    n_top_genes=1000,
)

print(f".X (raw counts): {result.X.shape}")
print(f"Layers: {list(result.layers.keys())}")
print(f".obsm keys: {list(result.obsm.keys())}")
print(f"PCA shape: {result.obsm['X_pca'].shape}")

## 3. Perturbation Embeddings Only

When you want to embed the perturbation identifiers (gene symbols,
SMILES, etc.) without computing cell embeddings from expression,
pass `perturbation_models` and set `preprocessing="none"`.

In [ ]:
result_pert = embedder.embed_adata(
    adata,
    perturbation_models=["esm2_650M"],
    perturbation_column="perturbation",
    perturbation_type="symbol",
    preprocessing="none",
)

print(f".obsm keys: {list(result_pert.obsm.keys())}")
print(f"ESM2 shape: {result_pert.obsm['X_esm2_650M'].shape}")

# Check metadata
meta = result_pert.uns["embpy_embeddings"]["esm2_650M"]
print(f"Perturbations embedded: {meta['n_perturbations_embedded']}/{meta['n_perturbations_total']}")
print(f"Cells mapped: {meta['n_cells_mapped']}/{meta['n_cells']}")

## 4. Combined Cell + Perturbation Embeddings

The real power of `embed_adata()` is computing everything in one call.
Cell embeddings and perturbation embeddings are stored side by side
in `.obsm`.

In [ ]:
result_combined = embedder.embed_adata(
    adata,
    # Cell-level
    cell_models=["pca"],
    preprocessing="standard",
    n_pca_components=30,
    n_top_genes=1000,
    # Perturbation-level
    perturbation_models=["esm2_650M"],
    perturbation_column="perturbation",
    perturbation_type="symbol",
)

print("All .obsm keys:")
for key in sorted(result_combined.obsm.keys()):
    print(f"  {key}: {result_combined.obsm[key].shape}")

print(f"\n.X (raw counts): {result_combined.X.shape}")
print(f"Layers: {list(result_combined.layers.keys())}")
print(f"\nMetadata keys: {list(result_combined.uns['embpy_embeddings'].keys())}")

## 5. Auto-Detecting Perturbation Types

Set `perturbation_type="auto"` to let embpy auto-detect whether each
perturbation identifier is a gene symbol, Ensembl ID, or SMILES string.
This is useful for datasets with mixed perturbation types.

In [ ]:
from embpy.resources.gene_resolver import detect_identifier_type

examples = ["TP53", "ENSG00000141510", "CCO", "BRCA1"]
for ident in examples:
    print(f"  {ident:25s} -> {detect_identifier_type(ident)}")

## 6. Inspecting Embedding Metadata

After `embed_adata()`, all metadata is stored in
`adata.uns["embpy_embeddings"]`. This includes embedding dimensions,
success counts, and the wrapper class used.

In [ ]:
import json

if "embpy_embeddings" in result_combined.uns:
    for model_key, info in result_combined.uns["embpy_embeddings"].items():
        print(f"\n{model_key}:")
        for k, v in info.items():
            print(f"  {k}: {v}")

## Summary

| What | How |
|---|---|
| Cell embeddings only | `embed_adata(adata, cell_models=["pca", "scvi"])` |
| Perturbation embeddings only | `embed_adata(adata, perturbation_models=["esm2_650M"], perturbation_column="gene")` |
| Both in one call | `embed_adata(adata, cell_models=[...], perturbation_models=[...], perturbation_column="...")` |
| Auto-detect perturbation type | `perturbation_type="auto"` |
| Results location | `.obsm["X_{model_name}"]` |
| Metadata | `.uns["embpy_embeddings"]` |